# CSD Sampling Experiment: Multi-Step Arithmetic × 3 LLMs × CSD Indicator Extraction

This notebook demonstrates the **Critical Slowing Down (CSD) sampling experiment**, which tests whether ecological early-warning signals can predict approaching reasoning collapse in LLMs.

**What this experiment does:**
- Samples LLM responses across 24 difficulty levels of multi-step arithmetic for 3 models (LLaMA-3.1-8B, Gemini-2.0-Flash, GPT-4o-mini)
- Computes 9 CSD indicators per (model, difficulty) pair: embedding variance, Hartigan dip test, silhouette k=2, bimodality coefficient, disagreement rate, extraction rate, step-correctness autocorrelation
- Fits variance scaling laws and tests for flickering as a leading indicator of capability collapse
- Key finding: d* (capability boundary) varies across models; variance trends are significant (p<0.02) for all models

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# All packages used are pre-installed on Colab; install locally only
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scipy==1.15.3', 'matplotlib==3.10.0', 'tabulate==0.9.0')

In [ ]:
import json
import math
import os
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats as sp_stats
from tabulate import tabulate

warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/ai-inventor-outputs/ai-invention-276cb0-flickering-before-failing-ecological-cri/main/experiment_iter2_csd_sampling_ex/demo/mini_demo_data.json"

def load_data():
    """Load demo data from GitHub (primary) or local file (fallback)."""
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
print(f"Loaded {len(data['datasets'])} datasets, "
      f"{sum(len(ds['examples']) for ds in data['datasets'])} total examples")
print(f"Models: {data['metadata']['models']}")
print(f"Difficulty levels: {data['metadata']['difficulty_levels']}")

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
# Analysis thresholds (from original method.py)
ACCURACY_THRESHOLD_DSTAR = 0.5       # accuracy below this → capability boundary d*
ACCURACY_THRESHOLD_LEADING = 0.80    # accuracy above this for leading-indicator test
FLICKERING_PVALUE_THRESHOLD = 0.05   # dip p-value below this → flickering detected
SILHOUETTE_THRESHOLD = 0.3           # silhouette above this → bimodal structure
BC_THRESHOLD = 0.555                 # bimodality coefficient above this → bimodal
MIN_POINTS_FOR_FIT = 3               # minimum data points for scaling law regression
# Original values: all identical (these ARE the original experiment thresholds)

## Phase 1: Parse CSD Indicators from Pre-computed Data

The experiment already sampled 3582 LLM responses, embedded them, and computed 9 CSD indicators per (model, difficulty) pair. Here we parse those pre-computed indicators into a structured DataFrame.

In [ ]:
# Parse all examples into a flat list of records
records = []
for ds in data["datasets"]:
    for ex in ds["examples"]:
        rec = {
            "model": ex["metadata_model"],
            "model_short": ex["metadata_model"].split("/")[-1],
            "difficulty": ex["metadata_difficulty_level"],
            "accuracy": float(ex["predict_accuracy"]),
            "csd_variance": float(ex["predict_csd_variance"]),
            "dip_statistic": float(ex["predict_dip_statistic"]),
            "dip_pvalue": float(ex["predict_dip_pvalue"]),
            "silhouette_k2": float(ex["predict_silhouette_k2"]),
            "bimodality_coefficient": float(ex["predict_bimodality_coefficient"]),
            "disagreement_rate": float(ex["predict_disagreement_rate"]),
            "extraction_rate": float(ex["predict_extraction_rate"]),
            "step_correctness_autocorr": float(ex["predict_step_correctness_autocorr"]),
            "num_responses": ex["metadata_num_responses"],
        }
        records.append(rec)

df = pd.DataFrame(records)
models = df["model"].unique().tolist()
levels = sorted(df["difficulty"].unique().tolist())
print(f"Parsed {len(df)} records: {len(models)} models × {len(levels)} difficulty levels")
df.head()

## Phase 2: Compute Capability Boundary (d*) per Model

The capability boundary **d*** is defined as the first difficulty level where accuracy drops below 50%. This is the "tipping point" — the ecological analogy to a regime shift.

In [ ]:
# Determine d* (capability boundary) = first level where accuracy < ACCURACY_THRESHOLD_DSTAR
d_star = {}
for model in models:
    mdf = df[df["model"] == model].sort_values("difficulty")
    found = False
    for _, row in mdf.iterrows():
        if row["accuracy"] < ACCURACY_THRESHOLD_DSTAR:
            d_star[model] = int(row["difficulty"])
            found = True
            break
    if not found:
        d_star[model] = levels[-1] + 1

for model in models:
    short = model.split("/")[-1]
    print(f"  {short}: d* = {d_star[model]}")

## Phase 3: Variance Scaling Law Fit

Near a phase transition, CSD theory predicts that response variance should increase as a power law: `Var ~ (d* - d)^α`. We fit `log(Var) ~ α·log(d* - d)` via linear regression for each model.

In [ ]:
# Fit variance scaling: log(Var) ~ alpha * log(d* - d)
scaling_fits = {}
for model in models:
    mdf = df[df["model"] == model].sort_values("difficulty")
    ds = d_star[model]
    log_dist, log_var = [], []
    for _, row in mdf.iterrows():
        lvl = row["difficulty"]
        var = row["csd_variance"]
        if lvl < ds and (ds - lvl) > 0 and var > 0:
            log_dist.append(math.log(ds - lvl))
            log_var.append(math.log(var))

    if len(log_dist) >= MIN_POINTS_FOR_FIT:
        slope, intercept, r_value, p_value, std_err = sp_stats.linregress(log_dist, log_var)
        scaling_fits[model] = {
            "exponent": float(slope),
            "r2": float(r_value ** 2),
            "intercept": float(intercept),
            "p_value": float(p_value),
        }
        short = model.split("/")[-1]
        print(f"  {short}: α = {slope:.4f}, R² = {r_value**2:.4f}, p = {p_value:.4f}")
    else:
        scaling_fits[model] = {"exponent": None, "r2": None, "intercept": None, "p_value": None}
        short = model.split("/")[-1]
        print(f"  {short}: insufficient data points for fit (need {MIN_POINTS_FOR_FIT}, got {len(log_dist)})")

## Phase 4: Leading Indicator Tests & Kendall Tau Trends

We test whether CSD signals appear **before** accuracy collapse (i.e., at levels where accuracy is still > 80%). Three indicators are checked: Hartigan dip (flickering), silhouette k=2, and bimodality coefficient. We also compute Kendall Tau rank correlations to test whether variance and dip statistics trend upward with difficulty.

In [ ]:
# Leading Indicator Tests + Kendall Tau Trend Tests (from original method.py Phase 6)
model_summary = {}
for model in models:
    mdf = df[df["model"] == model].sort_values("difficulty")
    short = model.split("/")[-1]

    # Flickering: dip_pvalue < threshold at any level where accuracy > ACCURACY_THRESHOLD_LEADING
    flickering_leading = False
    earliest_flickering = None
    for _, row in mdf.iterrows():
        if row["accuracy"] > ACCURACY_THRESHOLD_LEADING and row["dip_pvalue"] < FLICKERING_PVALUE_THRESHOLD:
            flickering_leading = True
            earliest_flickering = int(row["difficulty"])
            break

    # Silhouette > threshold at any level where accuracy > ACCURACY_THRESHOLD_LEADING
    sil_leading = False
    for _, row in mdf.iterrows():
        if row["accuracy"] > ACCURACY_THRESHOLD_LEADING and row["silhouette_k2"] > SILHOUETTE_THRESHOLD:
            sil_leading = True
            break

    # BC > threshold at any level where accuracy > ACCURACY_THRESHOLD_LEADING
    bc_leading = False
    for _, row in mdf.iterrows():
        if row["accuracy"] > ACCURACY_THRESHOLD_LEADING and row["bimodality_coefficient"] > BC_THRESHOLD:
            bc_leading = True
            break

    bimodality_consensus = sum([flickering_leading, sil_leading, bc_leading]) >= 2

    # Kendall Tau trend tests
    var_series = mdf["csd_variance"].values
    dip_series = mdf["dip_statistic"].values
    lvl_series = mdf["difficulty"].values

    if len(lvl_series) >= MIN_POINTS_FOR_FIT:
        tau_var, p_tau_var = sp_stats.kendalltau(lvl_series, var_series)
        tau_dip, p_tau_dip = sp_stats.kendalltau(lvl_series, dip_series)
    else:
        tau_var, p_tau_var = 0.0, 1.0
        tau_dip, p_tau_dip = 0.0, 1.0

    model_summary[model] = {
        "d_star": d_star[model],
        "scaling_exponent": scaling_fits[model]["exponent"],
        "scaling_r2": scaling_fits[model]["r2"],
        "flickering_leading": flickering_leading,
        "earliest_flickering_level": earliest_flickering,
        "sil_leading": sil_leading,
        "bc_leading": bc_leading,
        "bimodality_consensus_leading": bimodality_consensus,
        "tau_variance": float(tau_var) if not np.isnan(tau_var) else 0.0,
        "p_tau_variance": float(p_tau_var) if not np.isnan(p_tau_var) else 1.0,
        "tau_dip": float(tau_dip) if not np.isnan(tau_dip) else 0.0,
        "p_tau_dip": float(p_tau_dip) if not np.isnan(p_tau_dip) else 1.0,
    }

    print(f"  {short}: d*={d_star[model]}, flickering={flickering_leading}, "
          f"sil={sil_leading}, bc={bc_leading}, consensus={bimodality_consensus}")
    print(f"    τ_var={tau_var:.3f} (p={p_tau_var:.4f}), τ_dip={tau_dip:.3f} (p={p_tau_dip:.4f})")

## Results Summary

Summary table of all model-level findings: capability boundary d*, scaling exponent, leading indicators detected, and trend significance.

In [ ]:
# Print results summary table
rows = []
for model in models:
    ms = model_summary[model]
    short = model.split("/")[-1]
    exp_str = f"{ms['scaling_exponent']:.4f}" if ms['scaling_exponent'] is not None else "N/A"
    r2_str = f"{ms['scaling_r2']:.4f}" if ms['scaling_r2'] is not None else "N/A"
    sig_var = "✓" if ms["p_tau_variance"] < 0.05 else "✗"
    sig_dip = "✓" if ms["p_tau_dip"] < 0.05 else "✗"
    rows.append([
        short,
        ms["d_star"],
        exp_str,
        r2_str,
        "Yes" if ms["flickering_leading"] else "No",
        "Yes" if ms["sil_leading"] else "No",
        "Yes" if ms["bc_leading"] else "No",
        "Yes" if ms["bimodality_consensus_leading"] else "No",
        f"{ms['tau_variance']:.3f} {sig_var}",
        f"{ms['tau_dip']:.3f} {sig_dip}",
    ])

headers = ["Model", "d*", "α (exp)", "R²", "Flicker", "Sil", "BC", "Consensus",
           "τ_var (sig?)", "τ_dip (sig?)"]
print(tabulate(rows, headers=headers, tablefmt="github"))

## Visualization: CSD Indicators across Difficulty Levels

Four-panel plot showing how key CSD indicators evolve with increasing task difficulty for each model. Vertical dashed lines mark the capability boundary d* per model.

In [ ]:
colors = {"meta-llama/llama-3.1-8b-instruct": "#e74c3c",
          "google/gemini-2.0-flash-001": "#3498db",
          "openai/gpt-4o-mini": "#2ecc71"}

fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharex=True)
indicators_to_plot = [
    ("accuracy", "Accuracy", axes[0, 0]),
    ("csd_variance", "Embedding Variance (CSD)", axes[0, 1]),
    ("dip_statistic", "Hartigan Dip Statistic", axes[1, 0]),
    ("bimodality_coefficient", "Bimodality Coefficient", axes[1, 1]),
]

for col_name, ylabel, ax in indicators_to_plot:
    for model in models:
        mdf = df[df["model"] == model].sort_values("difficulty")
        short = model.split("/")[-1]
        c = colors[model]
        ax.plot(mdf["difficulty"], mdf[col_name], "o-", label=short,
                color=c, markersize=4, linewidth=1.5)
        # Mark d* with vertical dashed line
        ax.axvline(x=d_star[model], color=c, linestyle="--", alpha=0.5, linewidth=1)

    ax.set_ylabel(ylabel, fontsize=11)
    ax.grid(True, alpha=0.3)
    if col_name == "accuracy":
        ax.axhline(y=ACCURACY_THRESHOLD_DSTAR, color="gray", linestyle=":", alpha=0.7, label="d* threshold (0.5)")
        ax.axhline(y=ACCURACY_THRESHOLD_LEADING, color="gray", linestyle="-.", alpha=0.5, label="leading threshold (0.8)")
    if col_name == "bimodality_coefficient":
        ax.axhline(y=BC_THRESHOLD, color="gray", linestyle=":", alpha=0.7, label=f"BC threshold ({BC_THRESHOLD})")

for ax in axes[1]:
    ax.set_xlabel("Difficulty Level", fontsize=11)

axes[0, 0].legend(fontsize=8, loc="lower left")
fig.suptitle("CSD Indicators vs. Difficulty Level (dashed lines = d* per model)",
             fontsize=13, fontweight="bold", y=0.98)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

# ── Second figure: Silhouette, Disagreement Rate, Scaling Law ──
fig2, axes2 = plt.subplots(1, 3, figsize=(16, 4.5))

# Panel 1: Silhouette k=2
for model in models:
    mdf = df[df["model"] == model].sort_values("difficulty")
    short = model.split("/")[-1]
    c = colors[model]
    axes2[0].plot(mdf["difficulty"], mdf["silhouette_k2"], "o-", label=short,
                  color=c, markersize=4, linewidth=1.5)
    axes2[0].axvline(x=d_star[model], color=c, linestyle="--", alpha=0.5)
axes2[0].axhline(y=SILHOUETTE_THRESHOLD, color="gray", linestyle=":", alpha=0.7)
axes2[0].set_xlabel("Difficulty Level"); axes2[0].set_ylabel("Silhouette k=2")
axes2[0].set_title("Silhouette Score (k=2)"); axes2[0].grid(True, alpha=0.3)
axes2[0].legend(fontsize=8)

# Panel 2: Disagreement Rate
for model in models:
    mdf = df[df["model"] == model].sort_values("difficulty")
    short = model.split("/")[-1]
    c = colors[model]
    axes2[1].plot(mdf["difficulty"], mdf["disagreement_rate"], "o-", label=short,
                  color=c, markersize=4, linewidth=1.5)
    axes2[1].axvline(x=d_star[model], color=c, linestyle="--", alpha=0.5)
axes2[1].set_xlabel("Difficulty Level"); axes2[1].set_ylabel("Disagreement Rate")
axes2[1].set_title("Self-Consistency Disagreement"); axes2[1].grid(True, alpha=0.3)
axes2[1].legend(fontsize=8)

# Panel 3: Variance Scaling Law
for model in models:
    mdf = df[df["model"] == model].sort_values("difficulty")
    ds = d_star[model]
    short = model.split("/")[-1]
    c = colors[model]
    log_d, log_v = [], []
    for _, row in mdf.iterrows():
        lvl = row["difficulty"]
        var = row["csd_variance"]
        if lvl < ds and (ds - lvl) > 0 and var > 0:
            log_d.append(math.log(ds - lvl))
            log_v.append(math.log(var))
    if log_d:
        axes2[2].scatter(log_d, log_v, color=c, s=20, label=short, alpha=0.7)
        sf = scaling_fits[model]
        if sf["exponent"] is not None:
            x_fit = np.linspace(min(log_d), max(log_d), 50)
            y_fit = sf["exponent"] * x_fit + sf["intercept"]
            axes2[2].plot(x_fit, y_fit, "-", color=c, linewidth=1.5, alpha=0.8)
axes2[2].set_xlabel("log(d* − d)"); axes2[2].set_ylabel("log(Variance)")
axes2[2].set_title("Variance Scaling Law Fit"); axes2[2].grid(True, alpha=0.3)
axes2[2].legend(fontsize=8)

plt.tight_layout()
plt.show()
print("Visualization complete.")